In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df = pd.read_csv("dataset_analitico.csv")
df.columns = [c.lower().strip() for c in df.columns]
df = df.fillna(0)
print(df.head())

In [ ]:
media_col = "media_atraso"
taxa_col = "taxa_atraso"
valor_col = "valor_total"
liq_col = "indicador_liquidez_quantitativo_3m"

In [ ]:
scaler = MinMaxScaler()
media_norm = 1 - scaler.fit_transform(df[[media_col]]).flatten()
taxa_norm = 1 - scaler.fit_transform(df[[taxa_col]]).flatten()
liq_norm = scaler.fit_transform(df[[liq_col]]).flatten()
valor_norm = scaler.fit_transform(df[[valor_col]]).flatten()

In [ ]:
df["score_risco"] = (media_norm * 0.35 + taxa_norm * 0.35 + liq_norm * 0.20 + valor_norm * 0.10) * 100
df["score_risco"] = df["score_risco"].round(2)
print(df[["score_risco"]].head())

In [ ]:
def classificar(score):
    if score >= 75:
        return "A"
    elif score >= 50:
        return "B"
    return "C"

df["classificacao"] = df["score_risco"].apply(classificar)
print(df["classificacao"].value_counts())

In [ ]:
def nivel_risco(cl):
    if cl == "A":
        return "Baixo risco"
    elif cl == "B":
        return "Risco moderado"
    return "Risco crítico"

df["nivel_risco"] = df["classificacao"].apply(nivel_risco)

In [ ]:
def alerta(row):
    if row[media_col] >= df[media_col].quantile(0.95):
        return "ALERTA_ATRASO_EXTREMO"
    elif row[taxa_col] >= df[taxa_col].quantile(0.90):
        return "ALERTA_INADIMPLENCIA"
    elif row[valor_col] >= df[valor_col].quantile(0.99):
        return "ALERTA_CONCENTRACAO_FINANCEIRA"
    elif row["score_risco"] < 25:
        return "ALERTA_RISCO_CRITICO"
    return "NORMAL"

df["alerta_fraude"] = df.apply(alerta, axis=1)
print(df["alerta_fraude"].value_counts())

In [ ]:
def esg(score):
    if score >= 75:
        return "OURO"
    elif score >= 50:
        return "PRATA"
    return "BRONZE"

df["score_esg"] = df["score_risco"].apply(esg)
print(df["score_esg"].value_counts())

In [ ]:
def explicabilidade(row):
    if row["score_risco"] >= 75:
        return "Empresa apresentou estabilidade operacional, baixa inadimplência e bom perfil de liquidez."
    elif row["score_risco"] >= 50:
        return "Empresa apresentou risco moderado devido à recorrência parcial de atraso e oscilação financeira."
    return "Empresa apresentou recorrência elevada de atraso, inadimplência acima da média e baixa liquidez operacional."

df["explicabilidade"] = df.apply(explicabilidade, axis=1)
print(df[["explicabilidade"]].head())

In [ ]:
dataset_final = df[["id_pagador", "uf", "score_risco", "classificacao", "nivel_risco", "alerta_fraude", "score_esg", "explicabilidade", valor_col, media_col, taxa_col]].copy()
dataset_final = dataset_final.rename(columns={valor_col: "valor_total", media_col: "media_atraso", taxa_col: "taxa_atraso"})
print(dataset_final.head())
dataset_final.to_csv("RiskVision_Dataset_Final.csv", index=False)
print("Dataset exportado com sucesso!")